# Hackathon AI & ML — AutoGluon
**Stratégie** : laisser AutoGluon explorer automatiquement LightGBM, XGBoost, CatBoost, Random Forest, ExtraTree, KNN, et leur stacking multi-niveaux.  
**Métrique cible** : F1-score (classe 1).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

DATA_DIR      = Path('data')
GROUP_ID      = 'G1'
SUBMISSION_ID = '3'
TIME_LIMIT    = 3600  # secondes — augmenter si possible (ex: 7200 pour 2h)

print('AutoGluon prêt')

AutoGluon prêt


## 1. Chargement et préparation

In [2]:
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

# Split train / test
test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()
train_df   = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'Train : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

Train : (54064, 1541)  |  Test : (5000, 1540)
Taux de mortalité : 15.68%


In [3]:
# Features NaN par composante (même feature engineering que advanced.ipynb)
meta_cols  = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})
labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

def add_missing_features(df, labo_cols, exam_cols, quest_cols):
    all_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()
    df['feat_nb_missing_total'] = df[all_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_cols)
    labo_p  = [c for c in labo_cols  if c in df.columns]
    exam_p  = [c for c in exam_cols  if c in df.columns]
    quest_p = [c for c in quest_cols if c in df.columns]
    if labo_p:  df['feat_nb_missing_labo']  = df[labo_p].isnull().sum(axis=1)
    if exam_p:  df['feat_nb_missing_exam']  = df[exam_p].isnull().sum(axis=1)
    if quest_p: df['feat_nb_missing_quest'] = df[quest_p].isnull().sum(axis=1)
    return df

train_df  = add_missing_features(train_df,  labo_cols, exam_cols, quest_cols)
test_data = add_missing_features(test_data, labo_cols, exam_cols, quest_cols)

# Supprimer colonnes >70% manquant
MISSING_THRESHOLD = 0.70
missing_ratio = train_df.isnull().mean()
cols_to_drop  = [c for c in missing_ratio[missing_ratio > MISSING_THRESHOLD].index
                 if c not in ['SEQN', 'MORTSTAT_2019']]

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

print(f'Colonnes supprimées : {len(cols_to_drop)}')
print(f'Train final : {train_clean.shape}  |  Test final : {test_clean.shape}')

Colonnes supprimées : 1173
Train final : (54064, 370)  |  Test final : (5000, 369)


In [4]:
# Préparer les DataFrames pour AutoGluon
# AutoGluon attend : toutes les features + la colonne cible dans le DataFrame train
# SEQN est un ID — on le retire des features mais on garde pour la soumission

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
test_seqn_order = test_clean['SEQN']

train_ag = train_clean[feature_cols + ['MORTSTAT_2019']].copy()
test_ag  = test_clean[feature_cols].copy()

# AutoGluon veut la cible en string pour la classification
train_ag['MORTSTAT_2019'] = train_ag['MORTSTAT_2019'].astype(str)

print(f'Train AutoGluon : {train_ag.shape}')
print(f'Test AutoGluon  : {test_ag.shape}')

Train AutoGluon : (54064, 369)
Test AutoGluon  : (5000, 368)


## 2. Entraînement AutoGluon

`best_quality` : AutoGluon entraîne tous les modèles disponibles + stacking multi-niveaux.  
Ajuster `time_limit` selon le temps disponible (3600 = 1h, 7200 = 2h).

In [5]:
predictor = TabularPredictor(
    label='MORTSTAT_2019',
    eval_metric='f1',          # optimise directement le F1
    path='autogluon_models',   # sauvegarde les modèles sur disque
    verbosity=2,
)

predictor.fit(
    train_data=train_ag,
    time_limit=TIME_LIMIT,
    presets='best_quality',    # stacking multi-niveaux, tous les modèles
    num_bag_folds=5,           # bagging 5-fold = CV intégrée
    num_stack_levels=2,        # 2 niveaux de stacking
)

Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.1
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.4.0: Thu Mar 19 19:32:36 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T8103
CPU Count:          8
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       1.39 GB / 8.00 GB (17.4%)
Disk Space Avail:   23.64 GB / 205.45 GB (11.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=5, num_bag_sets=1
DyStack

[1000]	valid_set's binary_logloss: 0.224886	valid_set's f1: 0.690039
[1000]	valid_set's binary_logloss: 0.216929	valid_set's f1: 0.690186
[1000]	valid_set's binary_logloss: 0.220346	valid_set's f1: 0.694284
[2000]	valid_set's binary_logloss: 0.224624	valid_set's f1: 0.697136
[1000]	valid_set's binary_logloss: 0.211757	valid_set's f1: 0.694654


	0.6959	 = Validation score   (f1)
	158.63s	 = Training   runtime
	1.18s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 599.09s of the 2094.59s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.5.0`. 
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 598.56s of the 2094.07s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)
	To force training the model, specify the model h

[1000]	valid_set's binary_logloss: 0.219551	valid_set's f1: 0.697406
[1000]	valid_set's binary_logloss: 0.220803	valid_set's f1: 0.691759


	0.6944	 = Validation score   (f1)
	70.0s	 = Training   runtime
	0.64s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 526.88s of the 2022.38s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)
		Unable to import dependency torch
A quick tip is to install via `pip install torch`.
The minimum torch version is currently 2.6.
Fitting model: XGBoost_r33_BAG_L1 ... Training model for up to 526.37s of the 2021.87s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)
	To force training the mo

## 3. Résultats et comparaison des modèles

In [9]:
leaderboard = predictor.leaderboard(silent=True)
print(leaderboard[['model', 'score_val', 'pred_time_val', 'fit_time']].to_string())
print(f'\nMeilleur modèle : {leaderboard.iloc[0]["model"]}')
print(f'Meilleur F1 CV  : {leaderboard["score_val"].max():.4f}')
print(f'\nBaseline LGB    : 0.6945')

                       model  score_val  pred_time_val     fit_time
0          LightGBMXT_BAG_L1   0.697554       0.265519    35.311099
1        WeightedEnsemble_L4   0.697554       0.271005    39.113806
2        WeightedEnsemble_L2   0.697554       0.271948    36.730379
3             XGBoost_BAG_L3   0.696851      49.144217  2133.569343
4            LightGBM_BAG_L1   0.696567       0.204778    30.343528
5       LightGBM_r188_BAG_L1   0.696206       0.426653    81.072379
6       LightGBM_r131_BAG_L1   0.695901       1.181082   158.634931
7        WeightedEnsemble_L3   0.695652      42.830070  1812.303165
8       CatBoost_r137_BAG_L3   0.695517      48.773193  2123.443140
9       LightGBM_r131_BAG_L3   0.695458      49.150191  2197.093667
10       LightGBM_r96_BAG_L1   0.694378       0.639614    69.998165
11           LightGBM_BAG_L3   0.693733      48.790985  2134.969272
12         LightGBMXT_BAG_L2   0.693646      24.406801  1177.129966
13         LightGBMXT_BAG_L3   0.692875      48.

## 4. Soumission

In [ ]:
y_pred = predictor.predict(test_ag).astype(int)

submission = pd.DataFrame({
    'SEQN': test_seqn_order.values,
    'prediction': y_pred.values
}).sort_values('SEQN')

assert len(submission) == 5000, f'ERREUR : {len(submission)} lignes'

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)

print(f'Fichier : {filename}')
print(f'Lignes  : {len(submission)}')
print(f'Répartition : {submission["prediction"].value_counts().to_dict()}')

## 5. (Optionnel) Recharger les modèles sans réentraîner

In [ ]:
# Pour recharger sans réentraîner :
# predictor = TabularPredictor.load('autogluon_models')
# y_pred = predictor.predict(test_ag)
print('Décommenter pour recharger les modèles sauvegardés dans autogluon_models/')